
# Custom features

**Background.** Built-in extractors cover intensities and texture. When
your study needs its own voxel formula (here, liver DCE enhancement and
wash-out), you write a small extractor class, register it under a name,
and use that name in a ``Spec`` like any built-in one.

**Purpose.** You get a registered ``dce_hemodynamics`` extractor, a map
of arterial relative enhancement, the column means / SDs before and after
z-score, and habitats fitted inside each subject from the three maps.

**When to use.** When the columns you want are not a built-in extractor
and a formula string (:doc:`/auto_examples/02_stages/plot_02_expression`)
is not enough.

**Key terms.**

* **voxel feature** / **extract** -- see
  :doc:`/auto_examples/02_stages/plot_06_voxel_intensities`.
* **registry** -- HABIT's lookup table from a name (``"dce_hemodynamics"``)
  to a component class; ``Spec("dce_hemodynamics", ...)`` finds the class
  through it.
* **relative enhancement / wash-out** -- signal gain over the unenhanced
  phase, and signal loss after the arterial phase, each as a ratio.

Three liver DCE maps: arterial relative enhancement, arterial-to-portal
wash-out, and arterial-to-delayed wash-out.


## Register the DCE extractor
Three formulas become the columns clustering sees. The class is
registered under the name used by ``Spec("dce_hemodynamics")`` below.



In [ ]:
from pathlib import Path
from typing import Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np

from habit.contracts import VoxelFeatureField, cohort_from_directory
from habit.contracts.subject import Subject
from habit.datasets import fetch_demo
from habit.feature_preprocessing import ZScoreScaling
from habit.spec import HabitatSpec, Spec, Stage
from habit.spec.specs import Spec as ComponentSpec
from habit.viz import plot_habitat_overlay, plot_voxel_texture_slice
from habit.voxel_features import (
    VoxelFeatureExtractorRegistry,
    aligned_image,
    build_voxel_field,
    roi_voxels,
)
import habit.recipes as recipes

# Unenhanced, late-arterial, portal-venous, delayed (demo pack keys).
PHASES: Tuple[str, ...] = ("pre_contrast", "LAP", "PVP", "delay_3min")
ROI = "LAP"
DCE_FEATURES = {
    "relative_enhancement_lap": "(LAP - pre_contrast) / (pre_contrast + eps)",
    "relative_washout_pvp": "(LAP - PVP) / (LAP - pre_contrast + eps)",
    "relative_washout_delay": "(LAP - delay_3min) / (LAP - pre_contrast + eps)",
}


@VoxelFeatureExtractorRegistry.register("dce_hemodynamics")
class DCEHemodynamics:
    """Per-voxel DCE maps from four liver phases."""

    def __init__(
        self,
        phases: Sequence[str] = PHASES,
        roi: Optional[str] = None,
        eps: float = 1e-8,
    ) -> None:
        if len(phases) != 4:
            raise ValueError(
                "dce_hemodynamics expects four phases: unenhanced, "
                "arterial, portal, delayed."
            )
        self.phases: Tuple[str, ...] = tuple(phases)
        self.roi = roi
        self.eps = float(eps)

    @property
    def spec(self) -> ComponentSpec:
        """Return the algorithm specification used for provenance."""
        return ComponentSpec(
            name="dce_hemodynamics",
            params={
                "phases": list(self.phases),
                "roi": self.roi,
                "eps": self.eps,
            },
        )

    def __call__(self, subject: Subject) -> VoxelFeatureField:
        """Compute the three DCE columns inside the ROI."""
        # Locate the ROI voxels once; every phase is read at these positions.
        mask, inside, voxel_index = roi_voxels(subject, self.roi)
        owner = "dce_hemodynamics"
        pre, lap, pvp, delay = (
            aligned_image(subject, phase, mask, owner=owner)[inside]
            for phase in self.phases
        )
        values = np.column_stack(
            [
                (lap - pre) / (pre + self.eps),
                (lap - pvp) / (lap - pre + self.eps),
                (lap - delay) / (lap - pre + self.eps),
            ]
        ).astype(np.float64, copy=False)
        # Wrap the values with their grid positions and this extractor's
        # Spec, so the field can be drawn back into the image and traced.
        return build_voxel_field(
            subject,
            mask,
            voxel_index,
            tuple(DCE_FEATURES),
            values,
            self.spec,
        )

## Load two subjects



In [ ]:
DATA = fetch_demo()
cohort = cohort_from_directory(DATA, modalities=PHASES, roi=ROI)[:2]
subject = cohort[0]

## Arterial enhancement map
The extractor returns a :class:`~habit.contracts.VoxelFeatureField`,
which :func:`~habit.viz.plot_voxel_texture_slice` draws directly.



In [ ]:
dce_field = DCEHemodynamics(phases=PHASES, roi=ROI)(subject)
fig_map = plot_voxel_texture_slice(
    dce_field,
    feature="relative_enhancement_lap",
    anatomy=subject.image("LAP"),
    roi_mask=subject.mask(ROI),
    cmap="inferno",
    axis=0,
    crop_to="roi",
    feature_label="Relative enhancement",
)
Path("out").mkdir(exist_ok=True)
fig_map.savefig("out/custom_voxel_dce_map.png", dpi=150, bbox_inches="tight")
plt.show()

before = dce_field.feature_frame()
print("before zscore (mean / std):")
print(before.agg(["mean", "std"]).round(4))
print(before.head())
# Ratios live on different scales; z-score each column to mean 0 / SD 1
# so no single map dominates the k-means distance.
scaler = ZScoreScaling(across_features=False)
after = scaler.transform(before, scaler.fit(before))
print("after zscore (mean / std):")
print(after.agg(["mean", "std"]).round(4))
print(after.head())

## Cluster the three DCE maps
extract uses the extractor registered above. preprocess is per-subject
z-score. There is no pool, so fit runs inside each subject.



In [ ]:
spec = HabitatSpec(
    name="dce_hemodynamics_demo",
    stages=(
        # extract: the three formulas registered as dce_hemodynamics.
        Stage(
            "extract_voxel_features",
            Spec("dce_hemodynamics", {"phases": list(PHASES), "roi": ROI}),
        ),
        # preprocess: z-score each column on this subject.
        Stage("preprocess", Spec("zscore", {"across_features": False})),
        # fit: elbow between 2 and 5 habitats, per subject (no pool).
        Stage(
            "fit",
            Spec(
                "kmeans",
                {
                    "min_habitats": 2,
                    "max_habitats": 5,
                    "validation": "elbow",
                    "n_init": 3,
                },
            ),
        ),
        Stage("assign", Spec("nearest_centroid")),
        Stage("quantify", Spec("volume")),
    ),
    random_seed=21,
)
result = recipes.Study(spec=spec).fit_predict(cohort)
for sid, model in result.subject_models.items():
    print(f"{sid}: n_habitats={model.n_habitats}")
labels = np.unique(result.habitat_maps[0].label_array)
print("overlay labels", [int(v) for v in labels if v > 0])
fig = plot_habitat_overlay(
    subject.image("LAP"),
    result.habitat_maps[0],
    title="habitats",
)
Path("out").mkdir(exist_ok=True)
fig.savefig("out/custom_voxel_overlay.png", dpi=150, bbox_inches="tight")
plt.show()